In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install torch torchvision pdf2image flask
!apt-get install poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
import torch
import torchvision
import flask
import pdf2image

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision import datasets
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn
import torch.optim as optim
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir = "/content/drive/MyDrive/datasetOfHandwritings"

# Image Transform
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

dataset = datasets.ImageFolder(data_dir, transform=transform)

train_loader = torch.utils.data.DataLoader(
    dataset, batch_size=16, shuffle=True
)

class_names = dataset.classes
print("Writers:", class_names)

# Load Pretrained Model (Updated method)
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 15

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss:.4f} Accuracy: {accuracy:.2f}%")

print("Training Complete")

# Save Model
torch.save(model.state_dict(), "/content/drive/MyDrive/writer_model.pth")

# Save class names
with open("/content/drive/MyDrive/classes.json", "w") as f:
    json.dump(class_names, f)

print("Model Saved Successfully")


Writers: ['himavarsha', 'mamatha', 'renuka', 'sharath']
Epoch [1/15] Loss: 9.5712 Accuracy: 90.24%
Epoch [2/15] Loss: 6.9616 Accuracy: 93.81%
Epoch [3/15] Loss: 8.3970 Accuracy: 93.43%
Epoch [4/15] Loss: 5.4586 Accuracy: 94.93%
Epoch [5/15] Loss: 3.1310 Accuracy: 97.19%
Epoch [6/15] Loss: 2.2221 Accuracy: 98.31%
Epoch [7/15] Loss: 0.9873 Accuracy: 99.25%
Epoch [8/15] Loss: 1.5986 Accuracy: 98.50%
Epoch [9/15] Loss: 0.8414 Accuracy: 99.25%
Epoch [10/15] Loss: 0.8531 Accuracy: 99.25%
Epoch [11/15] Loss: 0.2801 Accuracy: 100.00%
Epoch [12/15] Loss: 0.3695 Accuracy: 99.62%
Epoch [13/15] Loss: 0.1380 Accuracy: 100.00%
Epoch [14/15] Loss: 0.5641 Accuracy: 99.44%
Epoch [15/15] Loss: 1.3261 Accuracy: 98.69%
Training Complete
Model Saved Successfully
